In [ ]:
# Packages

# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math
import seaborn as sns

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    
    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'BLS Data')
    path_main = os.path.join(path_sp, 'Data')
    
if user in ['jchoy', 'aazawii']:
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'BLS', 'config')



print(user)
print(path_git)

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

In [ ]:
# Set indicator
indicator_name = 'Jobs_1'

# Impoting data
# df_jobs = pd.read_csv(os.path.join(path_agol, indicator_name, indicator_name + '_MSA_BLS_SMU.csv'))
df_msa = pd.read_excel(os.path.join(path_main, 'Vibrant and Inclusive Places', 'Economy', 'Jobs', 'Jobs_1 Total', indicator_name + ' MSA BLS SMU.xlsx'     ), sheet_name = 'MSA'     )
df_nat = pd.read_excel(os.path.join(path_main, 'Vibrant and Inclusive Places', 'Economy', 'Jobs', 'Jobs_1 Total', indicator_name + ' National BLS CEU.xlsx'), sheet_name = 'National')

df_jobs = pd.concat([df_msa, df_nat])
df_jobs

## Month to Month

In [ ]:
df_plot = df_jobs.copy()

df_plot = df_plot[df_plot['Variable'] == 'All'].reset_index(drop = True)
df_plot = df_plot.drop(['Variable', 'Percentage'], axis = 1)


df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == '2000-01-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])


# SACOG roll up
conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)
wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])

df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate'})


df_plot

In [ ]:
fig = px.line(df_plot, x = 'date_', y = 'Growth Rate', color = 'Groups', template = 'plotly_white')

fig.update_yaxes(tick0=0, dtick=1)
fig.update_layout(legend_title=None, title='Monthly Job Growth Comparison: Sacramento and other Mid-Sized Metro Areas')

fig.show()

## January to January

In [ ]:
df_plot = df_jobs.copy()

df_plot = df_plot[df_plot['Variable'] == 'All']
df_plot = df_plot.drop(['Variable', 'Percentage'], axis = 1)


df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot = df_plot[df_plot['date_'].str.contains('01-01')].reset_index(drop = True)
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == '2000-01-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average

# SACOG roll up
conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])


# Set up groups
conditions = [   
         df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2000, 2007, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2008, 2011, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2012, 2020, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2021, 2021, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2022, 2024, 1)))
             ]
choices = ["Pre Recession", "Recession", "Post Recession", "Covid", "Post Covid"]
# choices = ["Pre Recession (2000-2008)", "Recession (2008-2011)", "Post Recession (2011-2020)", "Covid (2020)", "Post Covid (2020-2024)"]
# Jared to fix year labels for plot below

df_plot["Period"] = np.select(conditions, choices)

df_plot = df_plot.dropna()

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"])
df_plot1 = df_plot.groupby(['Groups', 'Period'], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2 = df_plot.groupby(['Groups'          ], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2['Period'] = 'Total'
df_plot = pd.concat([df_plot1, df_plot2])

df_plot['Period_Sort'] = pd.Categorical(df_plot['Period'], ["Total", "Pre Recession", "Recession", "Post Recession", "Covid", "Post Covid"])
df_plot = df_plot.sort_values(by = ['Groups', 'Period_Sort'], ascending = [True, True])
df_plot = df_plot.drop(['Period_Sort'], axis = 1)

df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate', 'Period':'Time Period'})

df_plot

In [ ]:
## Jared to fix the labels on the plot

fig = px.bar(df_plot, x = 'Time Period', y = 'Growth Rate', color = 'Groups', barmode = 'group', template = 'plotly_white')

fig.update_yaxes(tick0=0, dtick=1)
fig.update_layout(legend_title=None, title='Annual Job Growth Comparison: Sacramento and other Mid-Sized Metro Areas (January)')

fig.show()

## September to September

In [ ]:
df_plot = df_jobs.copy()

df_plot = df_plot[df_plot['Variable'] == 'All']
df_plot = df_plot.drop(['Variable', 'Percentage'], axis = 1)


df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot = df_plot[df_plot['date_'].str.contains('09-01')].reset_index(drop = True)
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == '2000-09-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average

# SACOG roll up
conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])


# Set up groups
conditions = [   
         df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2000, 2007, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2008, 2011, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2012, 2019, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2020, 2020, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2021, 2024, 1)))
             ]
choices = ["Pre Recession", "Recession", "Post Recession", "Covid", "Post Covid"]
df_plot["Period"] = np.select(conditions, choices)

df_plot = df_plot.dropna()

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"])
df_plot1 = df_plot.groupby(['Groups', 'Period'], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2 = df_plot.groupby(['Groups'          ], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2['Period'] = 'Total'
df_plot = pd.concat([df_plot1, df_plot2])

df_plot['Period_Sort'] = pd.Categorical(df_plot['Period'], ["Total", "Pre Recession", "Recession", "Post Recession", "Covid", "Post Covid"])
df_plot = df_plot.sort_values(by = ['Groups', 'Period_Sort'], ascending = [True, True])
df_plot = df_plot.drop(['Period_Sort'], axis = 1)

df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate', 'Period':'Time Period'})

df_plot

In [ ]:
# px.bar(df_plot, x = 'Period', y = 'Jobs_GR', facet_col = 'Groups')
fig = px.bar(df_plot, x = 'Time Period', y = 'Growth Rate', color = 'Groups', barmode = 'group', template = 'plotly_white')

fig.update_yaxes(tick0=0, dtick=1)
fig.update_layout(legend_title=None, title='Annual Job Growth Comparison: Sacramento and other Mid-Sized Metro Areas (September)')

fig.show()

## By Presidential Administration

In [ ]:
df_plot = df_jobs.copy()

df_plot = df_plot[df_plot['Variable'] == 'All']
df_plot = df_plot.drop(['Variable', 'Percentage'], axis = 1)


df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot = df_plot[df_plot['date_'].str.contains('01-01')].reset_index(drop = True)
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == '2000-01-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"]) # weighted average

# SACOG roll up
conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])


# Set up groups
conditions = [   
         df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2001, 2009, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2010, 2017, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2018, 2021, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2022, 2024, 1)))
             ]
choices = ["Bush", "Obama", "Trump", "Biden"]
df_plot["Administration"] = np.select(conditions, choices)

df_plot = df_plot.dropna()

wm = lambda x: np.average(x, weights = df_plot.loc[x.index, "Value"])
df_plot1 = df_plot.groupby(['Groups', 'Administration'], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2 = df_plot.groupby(['Groups'          ], as_index = False)['Jobs_GR'].agg(np.mean)
df_plot2['Administration'] = 'Total'
df_plot = pd.concat([df_plot1, df_plot2])

df_plot['Period_Sort'] = pd.Categorical(df_plot['Administration'], ["Total", "Bush", "Obama", "Trump", "Biden"])
df_plot = df_plot.sort_values(by = ['Groups', 'Period_Sort'], ascending = [True, True])
df_plot = df_plot.drop(['Period_Sort'], axis = 1)

df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate'})

df_plot

In [ ]:
fig = px.bar(df_plot, x = 'Administration', y = 'Growth Rate', color = 'Groups', barmode = 'group', template = 'plotly_white')

fig.update_yaxes(tick0=0, dtick=1)
fig.update_layout(legend_title=None, title='Annual Job Growth Comparison: Sacramento and other Mid-Sized Metro Areas (by Presidential Administration)')

fig.show()

In [ ]:
df_plot = df_jobs.copy()

df_plot = df_plot[df_plot['Variable'] == 'All']
df_plot = df_plot.drop(['Variable', 'Percentage'], axis = 1)


df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, True])
df_plot = df_plot[df_plot['date_'].str.contains('01-01')].reset_index(drop = True)
df_plot['Jobs_GR'] = df_plot['Value'].pct_change()*100
df_plot.loc[df_plot['date_'] == '2000-01-01', 'Jobs_GR'] = np.nan
df_plot.loc[df_plot['Jobs_GR'] == np.inf, 'Jobs_GR'] = np.nan
df_plot = df_plot.sort_values(['Geography', 'date_'], ascending = [True, False])


# SACOG roll up
conditions = [   
       df_plot['Geography'].str.contains('Sac|Yuba')
    , ~df_plot['Geography'].str.contains('Sac|Yuba|National')
    ,  df_plot['Geography'].str.contains('National')
             ]
choices = ['SACOG', 'Peer MSA', 'National']
df_plot['Groups'] = np.select(conditions, choices)
df_plot = df_plot.groupby(['date_', 'Groups'], as_index = False).agg(Value = ('Value', 'sum'), Jobs_GR = ('Jobs_GR', wm))
df_plot = df_plot.sort_values(['Groups', 'date_'], ascending = [True, False])


# Set up groups
conditions = [   
         df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2001, 2009, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2010, 2017, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2018, 2021, 1)))
       , df_plot['date_'].str.contains("|".join(str(year) for year in sequence(2022, 2024, 1)))
             ]
choices = ["Bush", "Obama", "Trump", "Biden"]
df_plot["Administration"] = np.select(conditions, choices)

df_plot = df_plot.dropna()

df_plot = df_plot.sort_values(by = ['Groups', 'date_'], ascending = [True, True])

df_plot = df_plot.rename(columns = {'Jobs_GR':'Growth Rate'})

df_plot

In [ ]:
fig = px.line(df_plot, x = 'date_', y = 'Growth Rate', color = 'Groups', template = 'plotly_white')

fig.update_yaxes(tick0=0, dtick=1)
fig.update_layout(legend_title=None, title='Monthly Job Growth Comparison: Sacramento and other Mid-Sized Metro Areas')
fig.add_vline(x = '2008-01-01', line_dash = 'dash')
fig.add_vline(x = '2017-01-01', line_dash = 'dash')
fig.add_vline(x = '2021-01-01', line_dash = 'dash')
fig.add_hline(y = 0, line_dash = 'dash', line_color = 'gray')


fig.show()